# Digital Crime Investigation
## Notebook 01 — Data Cleaning & Validation

**Author:** Baskara Kresna Juniarto  
**Project:** Transaction Fraud & Anomaly Analytics  
**Scope:** Data Analyst Portfolio  

---
This notebook covers Phase 1 of the investigation pipeline:
- Load all four source tables
- Assess data quality (nulls, duplicates, type errors, out-of-range values)
- Apply cleaning transformations and document decisions
- Export cleaned data for downstream analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

DATA_DIR = Path('../data')
print('Data directory:', DATA_DIR.resolve())

## 1. Load Raw Data

In [ ]:
txn      = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['timestamp'])
users    = pd.read_csv(DATA_DIR / 'users.csv', parse_dates=['registration_date'])
devices  = pd.read_csv(DATA_DIR / 'devices.csv', parse_dates=['first_seen', 'last_seen'])
merchants= pd.read_csv(DATA_DIR / 'merchants.csv')

print(f'Transactions : {len(txn):,} rows  x  {txn.shape[1]} cols')
print(f'Users        : {len(users):,} rows  x  {users.shape[1]} cols')
print(f'Devices      : {len(devices):,} rows  x  {devices.shape[1]} cols')
print(f'Merchants    : {len(merchants):,} rows  x  {merchants.shape[1]} cols')

## 2. Schema Inspection

In [ ]:
txn.info()

In [ ]:
txn.head(10)

## 3. Missing Value Analysis

In [ ]:
def null_report(df, label):
    null_cnt = df.isnull().sum()
    null_pct = (null_cnt / len(df) * 100).round(2)
    report = pd.DataFrame({'null_count': null_cnt, 'null_pct': null_pct})
    report = report[report.null_count > 0].sort_values('null_count', ascending=False)
    print(f'\n=== {label} — Null Report ===')
    print(report if len(report) else 'No nulls found. ✓')

for df, name in [(txn, 'transactions'), (users, 'users'),
                 (devices, 'devices'), (merchants, 'merchants')]:
    null_report(df, name)

## 4. Duplicate Check

In [ ]:
print('Duplicate transaction_id:', txn.duplicated(subset='transaction_id').sum())
print('Duplicate user_id       :', users.duplicated(subset='user_id').sum())
print('Duplicate device_id     :', devices.duplicated(subset='device_id').sum())
print('Duplicate merchant_id   :', merchants.duplicated(subset='merchant_id').sum())

## 5. Categorical Value Audit

In [ ]:
EXPECTED = {
    'transaction_status': {'completed', 'pending', 'failed'},
    'payment_method'    : {'credit_card', 'debit_card', 'e-wallet', 'bank_transfer'},
    'category'          : {'food_beverage', 'e-commerce', 'transfer', 'retail',
                           'bills', 'travel', 'gaming', 'entertainment'},
}

for col, valid in EXPECTED.items():
    found   = set(txn[col].unique())
    invalid = found - valid
    print(f'{col}: {found}')
    if invalid:
        print(f'  ⚠ Unexpected values: {invalid}')
    else:
        print(f'  All values valid ✓')

## 6. Amount Range & Outlier Check

In [ ]:
print(txn['amount'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

# Flag non-positive amounts
bad_amounts = txn[txn['amount'] <= 0]
print(f'\nNon-positive amounts: {len(bad_amounts)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
txn['amount'].plot(kind='hist', bins=50, ax=axes[0],
                   color='steelblue', edgecolor='white',
                   title='Amount Distribution (full range)')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'Rp{x/1e6:.1f}M'))

txn[txn['amount'] <= txn['amount'].quantile(0.95)]['amount'].plot(
    kind='hist', bins=50, ax=axes[1],
    color='darkorange', edgecolor='white',
    title='Amount Distribution (below p95)')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(
    lambda x, _: f'Rp{x/1e3:.0f}K'))

for ax in axes:
    ax.set_xlabel('Amount (IDR)')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../images/01_amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Referential Integrity Check

In [ ]:
orphan_users     = txn[~txn['user_id'].isin(users['user_id'])]
orphan_devices   = txn[~txn['device_id'].isin(devices['device_id'])]
orphan_merchants = txn[~txn['merchant_id'].isin(merchants['merchant_id'])]

print(f'Orphan user_id     : {len(orphan_users)}')
print(f'Orphan device_id   : {len(orphan_devices)}')
print(f'Orphan merchant_id : {len(orphan_merchants)}')

## 8. Data Cleaning — Apply Transformations

In [ ]:
txn_clean = txn.copy()

# Ensure binary flags are integers
txn_clean['refund_flag']     = txn_clean['refund_flag'].astype(int)
txn_clean['chargeback_flag'] = txn_clean['chargeback_flag'].astype(int)

# Sort by user and time for downstream window calculations
txn_clean = txn_clean.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# Derived: transaction date and hour
txn_clean['txn_date'] = txn_clean['timestamp'].dt.date
txn_clean['txn_hour'] = txn_clean['timestamp'].dt.hour
txn_clean['txn_dow']  = txn_clean['timestamp'].dt.dayofweek  # 0=Mon

print('Cleaning complete.')
print(f'Shape after cleaning: {txn_clean.shape}')

## 9. Data Quality Summary

In [ ]:
checks = {
    'Total transactions'   : len(txn_clean),
    'Date range start'     : str(txn_clean['timestamp'].min()),
    'Date range end'       : str(txn_clean['timestamp'].max()),
    'Null values'          : txn_clean.isnull().sum().sum(),
    'Duplicate IDs'        : txn_clean.duplicated('transaction_id').sum(),
    'Non-positive amounts' : (txn_clean['amount'] <= 0).sum(),
    'Orphan user_id'       : (~txn_clean['user_id'].isin(users['user_id'])).sum(),
    'Refunded transactions': txn_clean['refund_flag'].sum(),
    'Chargebacks'          : txn_clean['chargeback_flag'].sum(),
}

summary_df = pd.DataFrame.from_dict(checks, orient='index', columns=['Value'])
print('\n=== DATA QUALITY SUMMARY ===')
print(summary_df.to_string())

## 10. Export Cleaned Data

In [ ]:
txn_clean.to_csv(DATA_DIR / 'transactions_clean.csv', index=False)
print('transactions_clean.csv written ✓')
print(f'  Rows: {len(txn_clean):,}  |  Cols: {txn_clean.shape[1]}')